In [0]:
# [Databricks Cell 1]
%pip install fastapi uvicorn structlog requests pydantic azure-monitor-opentelemetry

  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.39.1
    Not uninstalling opentelemetry-api at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-9d92d63a-dccd-4761-bda4-ead44ff6b99b
    Can't uninstall 'opentelemetry-api'. No files were found to uninstall.
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.60b1
    Not uninstalling opentelemetry-semantic-conventions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-9d92d63a-dccd-4761-bda4-ead44ff6b99b
    Can't uninstall 'opentelemetry-semantic-conventions'. No files were found to uninstall.
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.39.1
    Not uninstalling opentelemetry-sdk at /databricks/python3/lib/python3.12/site-packages, outsid

In [0]:
# [Databricks Cell 2 - UPDATED]
import os
import sys
import time
import uuid
import logging
import threading
import asyncio
import structlog
import uvicorn
from fastapi import FastAPI, status, HTTPException
from pydantic import BaseModel, Field

 

# ─── NEW: AZURE MONITOR PIPELINE CONFIGURATION ──────────────────────
from azure.monitor.opentelemetry import configure_azure_monitor

 

# Paste your copied string here (or use databricks secrets utility)
AZURE_CONN_STR = "InstrumentationKey=d8e51e0f-bddf-4adb-9947-f07045234d14;IngestionEndpoint=https://eastus-8.in.applicationinsights.azure.com/;LiveEndpoint=https://eastus.livediagnostics.monitor.azure.com/;ApplicationId=27403685-e31e-4816-816c-f2b883d5d635"

 

configure_azure_monitor(
    connection_string=AZURE_CONN_STR,
    logger_name="fastapi_routing_app" # Isolates app metrics from underlying SDK trace logs
)
# ─────────────────────────────────────────────────────────────────────

 

# 1. Structured Logging Configuration
logging.basicConfig(format="%(message)s", stream=sys.stdout, level=logging.INFO)
structlog.reset_defaults()
structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer() # Log Analytics can naturally parse this JSON string
    ],
    context_class=dict,
    logger_factory=structlog.stdlib.LoggerFactory(),
    wrapper_class=structlog.stdlib.BoundLogger,
    cache_logger_on_first_use=True,
)
logger = structlog.get_logger("fastapi_routing_app")

 

# 2. FastAPI Application Setup
app = FastAPI(title="Connected Azure Log Analytics App")

 

class TargetRequest(BaseModel):
    user_name: str
    payload: dict

 

def run_ml_inference(data: dict) -> dict:
    logger.info("Starting ML model execution...", approach="neural_network_inference")
    time.sleep(0.150)
    return {"predicted_class": 0, "confidence": 0.94}

 

async def run_simple_pipeline(data: dict) -> dict:
    logger.info("Executing standard database processing pipeline...")
    return {"status": "processed"}

 

@app.post("/execute", status_code=status.HTTP_200_OK)
async def route_user_request(request: TargetRequest):
    user_name = request.user_name
    # Explicit logging context tracking
    log = logger.bind(user_name=user_name, request_id=str(uuid.uuid4()))

 

    if "ml_user" in user_name:
        log.info("Routing request to ML inference pool")
        result = await asyncio.to_thread(run_ml_inference, request.payload)
        log.info("ML inference successfully completed")
        return {"routing": "ml_pipeline", "result": result}
    elif "db_user" in user_name:
        log.info("Routing request to standard database function")
        result = await run_simple_pipeline(request.payload)
        log.info("Database execution successfully completed")
        return {"routing": "database_pipeline", "result": result}
    else:
        raise HTTPException(status_code=400, detail="Invalid user naming convention.")



Exception occurred when importing Azure SDK Tracing.Please upgrade to the latest OpenTelemetry version: No module named 'azure.core.tracing.ext.opentelemetry_span'.
